In [1]:
import json
import os
import re
import shlex
import shutil
import subprocess
import sys
import time
from pathlib import Path
from urllib.parse import urlparse

try:
    import psutil
    PSUTIL_AVAILABLE = True
except ImportError:
    PSUTIL_AVAILABLE = False

try:
    import pandas as pd
    PANDAS_AVAILABLE = True
except ImportError:
    PANDAS_AVAILABLE = False

In [2]:
if not PSUTIL_AVAILABLE:
    print("⚠️ psutil is not installed.")
    print("Runtime can still be measured, but peak memory and CPU statistics will be limited.")
    print("To enable full measurements, run: pip install psutil")
else:
    print("✅ psutil available: runtime, CPU, and peak memory can be measured.")

✅ psutil available: runtime, CPU, and peak memory can be measured.


In [3]:
class ResourceEfficiencyChecker:
    """
    ResourceEfficiencyChecker

    This checker evaluates whether a research software artifact uses computational
    resources within predefined bounds.

    It supports two types of checks:

    1. Static repository footprint checks:
       - repository size
       - number of files
       - number of declared dependencies

    2. Optional benchmark execution checks:
       - wall-clock runtime
       - peak memory usage
       - CPU time

    The benchmark command can be provided in artifacts.json.

    Example artifact entry:

    {
      "title": "Example Artifact",
      "uri": "https://github.com/example/project",
      "description": "Example repository",
      "resource_efficiency": {
        "benchmark_command": "python main.py --help",
        "max_wall_time_seconds": 30,
        "max_peak_memory_mb": 512,
        "max_cpu_time_seconds": 60,
        "max_repo_size_mb": 500,
        "max_file_count": 50000,
        "max_dependency_count": 200
      }
    }
    """

    def __init__(
        self,
        json_file,
        download_dir="downloads",
        default_max_repo_size_mb=500,
        default_max_file_count=50000,
        default_max_dependency_count=200,
        default_timeout_seconds=60,
        require_benchmark=False
    ):
        self.json_file = json_file
        self.download_dir = Path(download_dir)
        self.download_dir.mkdir(parents=True, exist_ok=True)

        self.default_max_repo_size_mb = default_max_repo_size_mb
        self.default_max_file_count = default_max_file_count
        self.default_max_dependency_count = default_max_dependency_count
        self.default_timeout_seconds = default_timeout_seconds
        self.require_benchmark = require_benchmark

        self.artifacts = self.load_metadata(json_file)
        self.results = []

    def load_metadata(self, json_file):
        with open(json_file, "r", encoding="utf-8") as file:
            data = json.load(file)

        return data.get("artifacts", {})

    def is_git_repository(self, uri):
        return (
            isinstance(uri, str)
            and uri.startswith("https://github.com/")
        )

    def repo_name_from_uri(self, uri):
        parsed = urlparse(uri)
        repo_name = parsed.path.rstrip("/").split("/")[-1]

        if repo_name.endswith(".git"):
            repo_name = repo_name[:-4]

        return repo_name or "repository"

    def clone_repository(self, artifact_id, uri):
        repo_name = self.repo_name_from_uri(uri)
        target_dir = self.download_dir / repo_name

        if target_dir.exists():
            print(f"📁 Repository already exists: {target_dir}")
            return target_dir

        print(f"⬇️ Cloning repository: {uri}")

        try:
            result = subprocess.run(
                ["git", "clone", "--depth", "1", uri, str(target_dir)],
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                timeout=120
            )

            if result.returncode != 0:
                print(f"❌ Failed to clone repository for {artifact_id}")
                print(result.stderr.strip())
                return None

            print(f"✅ Cloned to: {target_dir}")
            return target_dir

        except Exception as e:
            print(f"❌ Clone error for {artifact_id}: {e}")
            return None

    def get_directory_size_mb(self, directory):
        total_size = 0

        for root, dirs, files in os.walk(directory):
            for file in files:
                file_path = Path(root) / file

                try:
                    total_size += file_path.stat().st_size
                except OSError:
                    pass

        return total_size / (1024 * 1024)

    def count_files(self, directory):
        file_count = 0

        for _, _, files in os.walk(directory):
            file_count += len(files)

        return file_count

    def parse_requirements_txt(self, requirements_path):
        dependencies = []

        try:
            with open(requirements_path, "r", encoding="utf-8", errors="ignore") as file:
                for line in file:
                    line = line.strip()

                    if not line or line.startswith("#"):
                        continue

                    if line.startswith("-r ") or line.startswith("--"):
                        continue

                    dependencies.append(line)

        except Exception:
            pass

        return dependencies

    def parse_environment_yml(self, environment_path):
        dependencies = []

        try:
            with open(environment_path, "r", encoding="utf-8", errors="ignore") as file:
                for line in file:
                    line = line.strip()

                    if not line or line.startswith("#"):
                        continue

                    if line.startswith("- "):
                        dep = line.replace("- ", "", 1).strip()

                        if dep and not dep.startswith("pip:"):
                            dependencies.append(dep)

        except Exception:
            pass

        return dependencies

    def parse_pyproject_toml(self, pyproject_path):
        dependencies = []

        try:
            with open(pyproject_path, "r", encoding="utf-8", errors="ignore") as file:
                content = file.read()

            dependency_blocks = re.findall(
                r"dependencies\s*=\s*\[(.*?)\]",
                content,
                flags=re.DOTALL
            )

            for block in dependency_blocks:
                matches = re.findall(r'"([^"]+)"|\'([^\']+)\'', block)

                for match in matches:
                    dep = match[0] or match[1]
                    if dep:
                        dependencies.append(dep)

        except Exception:
            pass

        return dependencies

    def collect_dependencies(self, repo_dir):
        dependencies = []

        requirements_file = repo_dir / "requirements.txt"
        environment_file = repo_dir / "environment.yml"
        pyproject_file = repo_dir / "pyproject.toml"

        if requirements_file.exists():
            dependencies.extend(self.parse_requirements_txt(requirements_file))

        if environment_file.exists():
            dependencies.extend(self.parse_environment_yml(environment_file))

        if pyproject_file.exists():
            dependencies.extend(self.parse_pyproject_toml(pyproject_file))

        cleaned = []
        seen = set()

        for dep in dependencies:
            dep_clean = dep.strip()

            if dep_clean and dep_clean not in seen:
                cleaned.append(dep_clean)
                seen.add(dep_clean)

        return cleaned

    def get_resource_config(self, artifact_data):
        config = artifact_data.get("resource_efficiency", {})
    
        return {
            "benchmark_command": config.get("benchmark_command", artifact_data.get("benchmark_command")),
            "dependencies": config.get("dependencies", []),
            "requirements_file": config.get("requirements_file"),
            "install_dependencies": config.get("install_dependencies", False),
            "max_wall_time_seconds": config.get("max_wall_time_seconds"),
            "max_peak_memory_mb": config.get("max_peak_memory_mb"),
            "max_cpu_time_seconds": config.get("max_cpu_time_seconds"),
            "max_repo_size_mb": config.get("max_repo_size_mb", self.default_max_repo_size_mb),
            "max_file_count": config.get("max_file_count", self.default_max_file_count),
            "max_dependency_count": config.get("max_dependency_count", self.default_max_dependency_count),
            "timeout_seconds": config.get("timeout_seconds", self.default_timeout_seconds),
        }

    def measure_process_resources(self, command, cwd, timeout_seconds):
        start_time = time.perf_counter()

        result = {
            "command": command,
            "executed": False,
            "timed_out": False,
            "return_code": None,
            "wall_time_seconds": None,
            "peak_memory_mb": None,
            "cpu_time_seconds": None,
            "stdout": "",
            "stderr": "",
            "error": None
        }

        try:
            if isinstance(command, list):
                process_command = command
                use_shell = False
            else:
                process_command = command
                use_shell = True

            process = subprocess.Popen(
                process_command,
                cwd=str(cwd),
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True,
                shell=use_shell
            )

            result["executed"] = True

            peak_memory_mb = 0.0
            cpu_time_seconds = 0.0

            if PSUTIL_AVAILABLE:
                ps_process = psutil.Process(process.pid)

                while process.poll() is None:
                    elapsed = time.perf_counter() - start_time

                    if elapsed > timeout_seconds:
                        result["timed_out"] = True
                        self.kill_process_tree(process.pid)
                        break

                    try:
                        processes = [ps_process] + ps_process.children(recursive=True)

                        total_memory = 0
                        total_cpu_time = 0.0

                        for p in processes:
                            try:
                                total_memory += p.memory_info().rss
                                cpu_times = p.cpu_times()
                                total_cpu_time += cpu_times.user + cpu_times.system
                            except Exception:
                                pass

                        peak_memory_mb = max(peak_memory_mb, total_memory / (1024 * 1024))
                        cpu_time_seconds = max(cpu_time_seconds, total_cpu_time)

                    except Exception:
                        pass

                    time.sleep(0.2)

            else:
                try:
                    process.wait(timeout=timeout_seconds)
                except subprocess.TimeoutExpired:
                    result["timed_out"] = True
                    self.kill_process_tree(process.pid)

            stdout, stderr = process.communicate(timeout=5)

            result["return_code"] = process.returncode
            result["stdout"] = stdout[-2000:] if stdout else ""
            result["stderr"] = stderr[-2000:] if stderr else ""

            end_time = time.perf_counter()

            result["wall_time_seconds"] = round(end_time - start_time, 4)

            if PSUTIL_AVAILABLE:
                result["peak_memory_mb"] = round(peak_memory_mb, 4)
                result["cpu_time_seconds"] = round(cpu_time_seconds, 4)

            return result

        except Exception as e:
            result["error"] = str(e)
            result["wall_time_seconds"] = round(time.perf_counter() - start_time, 4)
            return result

    def kill_process_tree(self, pid):
        try:
            if PSUTIL_AVAILABLE:
                parent = psutil.Process(pid)

                for child in parent.children(recursive=True):
                    child.kill()

                parent.kill()
            else:
                subprocess.run(
                    ["taskkill", "/F", "/T", "/PID", str(pid)],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE
                )

        except Exception:
            pass

    def evaluate_static_resource_footprint(self, repo_dir, config):
        repo_size_mb = self.get_directory_size_mb(repo_dir)
        file_count = self.count_files(repo_dir)
        dependencies = self.collect_dependencies(repo_dir)
        dependency_count = len(dependencies)

        size_ok = repo_size_mb <= config["max_repo_size_mb"]
        file_count_ok = file_count <= config["max_file_count"]
        dependency_count_ok = dependency_count <= config["max_dependency_count"]

        return {
            "repo_size_mb": round(repo_size_mb, 4),
            "file_count": file_count,
            "dependency_count": dependency_count,
            "dependencies": dependencies,
            "size_ok": size_ok,
            "file_count_ok": file_count_ok,
            "dependency_count_ok": dependency_count_ok,
            "static_resource_ok": size_ok and file_count_ok and dependency_count_ok
        }

    def evaluate_benchmark(self, repo_dir, config):
        command = config.get("benchmark_command")

        if not command:
            return {
                "benchmark_configured": False,
                "benchmark_ok": None,
                "reason": "No benchmark command provided."
            }

        print(f"🏃 Running benchmark command: {command}")

        measurement = self.measure_process_resources(
            command=command,
            cwd=repo_dir,
            timeout_seconds=config["timeout_seconds"]
        )

        wall_time_ok = True
        peak_memory_ok = True
        cpu_time_ok = True
        return_code_ok = measurement["return_code"] == 0
        timeout_ok = not measurement["timed_out"]

        if config["max_wall_time_seconds"] is not None and measurement["wall_time_seconds"] is not None:
            wall_time_ok = measurement["wall_time_seconds"] <= config["max_wall_time_seconds"]

        if config["max_peak_memory_mb"] is not None and measurement["peak_memory_mb"] is not None:
            peak_memory_ok = measurement["peak_memory_mb"] <= config["max_peak_memory_mb"]

        if config["max_cpu_time_seconds"] is not None and measurement["cpu_time_seconds"] is not None:
            cpu_time_ok = measurement["cpu_time_seconds"] <= config["max_cpu_time_seconds"]

        benchmark_ok = (
            return_code_ok
            and timeout_ok
            and wall_time_ok
            and peak_memory_ok
            and cpu_time_ok
        )

        measurement.update({
            "benchmark_configured": True,
            "return_code_ok": return_code_ok,
            "timeout_ok": timeout_ok,
            "wall_time_ok": wall_time_ok,
            "peak_memory_ok": peak_memory_ok,
            "cpu_time_ok": cpu_time_ok,
            "benchmark_ok": benchmark_ok
        })

        return measurement

    def check_artifact(self, artifact_id, artifact_data):
        title = artifact_data.get("title", "")
        uri = artifact_data.get("uri", "")
        config = self.get_resource_config(artifact_data)

        print("\n" + "=" * 80)
        print(f"🔍 Resource Efficiency Check for {artifact_id}")
        print(f"📦 Title: {title}")
        print(f"🔗 URI: {uri}")

        artifact_result = {
            "artifact_id": artifact_id,
            "title": title,
            "uri": uri,
            "resource_efficient": False,
            "status": "failed"
        }

        if not self.is_git_repository(uri):
            print("❌ Unsupported artifact type for this checker.")
            print("   Currently this notebook supports public GitHub repositories.")
            artifact_result["reason"] = "Unsupported artifact type."
            return artifact_result

        repo_dir = self.clone_repository(artifact_id, uri)

        if repo_dir is None:
            artifact_result["reason"] = "Repository could not be cloned."
            return artifact_result

        static_result = self.evaluate_static_resource_footprint(repo_dir, config)

        dependencies_ok = self.install_resource_dependencies(repo_dir, config)

        if dependencies_ok:
            benchmark_result = self.evaluate_benchmark(repo_dir, config)
        else:
            benchmark_result = {
                "benchmark_configured": True,
                "benchmark_ok": False,
                "reason": "Dependency installation failed.",
                "return_code": None,
                "timed_out": False,
                "wall_time_seconds": None,
                "peak_memory_mb": None,
                "cpu_time_seconds": None
            }

        artifact_result.update(static_result)
        artifact_result.update({
            "benchmark_configured": benchmark_result.get("benchmark_configured"),
            "benchmark_ok": benchmark_result.get("benchmark_ok"),
            "wall_time_seconds": benchmark_result.get("wall_time_seconds"),
            "peak_memory_mb": benchmark_result.get("peak_memory_mb"),
            "cpu_time_seconds": benchmark_result.get("cpu_time_seconds"),
            "return_code": benchmark_result.get("return_code"),
            "timed_out": benchmark_result.get("timed_out"),
        })

        print("\n📊 Static resource footprint:")
        print(f" - Repository size: {static_result['repo_size_mb']} MB / limit {config['max_repo_size_mb']} MB {'✅' if static_result['size_ok'] else '❌'}")
        print(f" - File count: {static_result['file_count']} / limit {config['max_file_count']} {'✅' if static_result['file_count_ok'] else '❌'}")
        print(f" - Dependency count: {static_result['dependency_count']} / limit {config['max_dependency_count']} {'✅' if static_result['dependency_count_ok'] else '❌'}")

        if benchmark_result.get("benchmark_configured"):
            print("\n📊 Benchmark resource usage:")
            print(f" - Return code: {benchmark_result.get('return_code')} {'✅' if benchmark_result.get('return_code_ok') else '❌'}")
            print(f" - Timed out: {benchmark_result.get('timed_out')} {'❌' if benchmark_result.get('timed_out') else '✅'}")
            print(f" - Wall time: {benchmark_result.get('wall_time_seconds')} seconds {'✅' if benchmark_result.get('wall_time_ok') else '❌'}")

            if benchmark_result.get("peak_memory_mb") is not None:
                print(f" - Peak memory: {benchmark_result.get('peak_memory_mb')} MB {'✅' if benchmark_result.get('peak_memory_ok') else '❌'}")
            else:
                print(" - Peak memory: not measured")

            if benchmark_result.get("cpu_time_seconds") is not None:
                print(f" - CPU time: {benchmark_result.get('cpu_time_seconds')} seconds {'✅' if benchmark_result.get('cpu_time_ok') else '❌'}")
            else:
                print(" - CPU time: not measured")

        else:
            print("\n⚠️ Benchmark resource usage was not evaluated.")
            print("   Reason: no benchmark_command was provided in artifacts.json.")

        static_ok = static_result["static_resource_ok"]

        if benchmark_result.get("benchmark_configured"):
            benchmark_ok = benchmark_result.get("benchmark_ok") is True
            resource_efficient = static_ok and benchmark_ok
            status = "passed" if resource_efficient else "failed"

        else:
            if self.require_benchmark:
                resource_efficient = False
                status = "failed_no_benchmark"
            else:
                resource_efficient = static_ok
                status = "partial_static_only"

        artifact_result["resource_efficient"] = resource_efficient
        artifact_result["status"] = status

        if status == "passed":
            print("\n✅ Resource Efficiency Result: PASSED")
        elif status == "partial_static_only":
            print("\n⚠️ Resource Efficiency Result: PARTIAL PASS")
            print("   Static footprint is within bounds, but no benchmark scenario was executed.")
        else:
            print("\n❌ Resource Efficiency Result: FAILED")

        return artifact_result

    def install_resource_dependencies(self, repo_dir, config):
        install_dependencies = config.get("install_dependencies", False)
    
        if not install_dependencies:
            print("ℹ️ Dependency installation skipped.")
            return True
    
        print("📦 Installing benchmark dependencies...")
    
        dependencies = config.get("dependencies", [])
        requirements_file = config.get("requirements_file")
    
        if requirements_file:
            requirements_path = Path(repo_dir) / requirements_file
    
            if requirements_path.exists():
                print(f"📄 Installing from requirements file: {requirements_file}")
    
                result = subprocess.run(
                    [sys.executable, "-m", "pip", "install", "-r", str(requirements_path)],
                    stdout=subprocess.PIPE,
                    stderr=subprocess.PIPE,
                    text=True
                )
    
                if result.returncode != 0:
                    print("❌ Failed to install requirements file.")
                    print(result.stderr)
                    return False
    
                print("✅ Requirements installed successfully.")
            else:
                print(f"❌ Requirements file not found: {requirements_file}")
                return False
    
        if dependencies:
            print("📦 Installing declared dependencies:")
            for dep in dependencies:
                print(f" - {dep}")
    
            result = subprocess.run(
                [sys.executable, "-m", "pip", "install"] + dependencies,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=True
            )
    
            if result.returncode != 0:
                print("❌ Failed to install declared dependencies.")
                print(result.stderr)
                return False
    
            print("✅ Declared dependencies installed successfully.")
    
        if not dependencies and not requirements_file:
            print("⚠️ install_dependencies is true, but no dependencies or requirements_file were specified.")
    
        return True

    def run(self):
        self.results = []

        print("🌱 Starting Resource Efficiency Fitness Function")
        print(f"📄 Metadata file: {self.json_file}")
        print(f"📁 Download directory: {self.download_dir}")

        for artifact_id, artifact_data in self.artifacts.items():
            result = self.check_artifact(artifact_id, artifact_data)
            self.results.append(result)

        print("\n" + "=" * 80)
        print("📌 Resource Efficiency Summary")
        print("=" * 80)

        for result in self.results:
            icon = "✅" if result["resource_efficient"] else "❌"
            print(f"{icon} {result['artifact_id']}: {result['status']}")

        return self.results

In [4]:
checker = ResourceEfficiencyChecker(
    json_file="artifacts.json",
    download_dir="downloads",
    default_max_repo_size_mb=500,
    default_max_file_count=50000,
    default_max_dependency_count=200,
    default_timeout_seconds=60,
    require_benchmark=False
)

results = checker.run()

🌱 Starting Resource Efficiency Fitness Function
📄 Metadata file: artifacts.json
📁 Download directory: downloads

🔍 Resource Efficiency Check for artifact_1
📦 Title: We provide our resources in a dedicated repository
🔗 URI: https://github.com/hihey54/hicss58
📁 Repository already exists: downloads/hicss58
📦 Installing benchmark dependencies...
📦 Installing declared dependencies:
 - requests>=2.31.0
 - beautifulsoup4>=4.12.3
 - PyMuPDF>=1.23.20
✅ Declared dependencies installed successfully.
🏃 Running benchmark command: python CODE/user_study_pdfs.py && python CODE/technical_pdfs.py && python CODE/user_study_repo.py && python CODE/technical_repo.py

📊 Static resource footprint:
 - Repository size: 1.5753 MB / limit 500 MB ✅
 - File count: 48 / limit 50000 ✅
 - Dependency count: 3 / limit 200 ✅

📊 Benchmark resource usage:
 - Return code: 0 ✅
 - Timed out: False ✅
 - Wall time: 0.415 seconds ✅
 - Peak memory: 10.7891 MB ✅
 - CPU time: 0.0 seconds ✅

✅ Resource Efficiency Result: PASSED

🔍 

In [5]:
if PANDAS_AVAILABLE:
    df = pd.DataFrame(results)

    columns_to_show = [
        "artifact_id",
        "title",
        "resource_efficient",
        "status",
        "repo_size_mb",
        "file_count",
        "dependency_count",
        "benchmark_configured",
        "wall_time_seconds",
        "peak_memory_mb",
        "cpu_time_seconds",
        "return_code",
        "timed_out"
    ]

    existing_columns = [col for col in columns_to_show if col in df.columns]

    display(df[existing_columns])
else:
    for result in results:
        print(result)

,artifact_id,title,resource_efficient,status,repo_size_mb,file_count,dependency_count,benchmark_configured,wall_time_seconds,peak_memory_mb,cpu_time_seconds,return_code,timed_out
0,artifact_1,We provide our resources in a dedicated reposi...,True,passed,1.5753,48.0,3.0,True,0.4150,10.7891,0.0,0.0,False
1,artifact_2,Trending Customer Dataset,False,failed,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,artifact_3,Python algorithms,True,passed,23.4867,1909.0,21.0,True,0.2058,1.8242,0.0,0.0,False
3,artifact_4,Scikit-learn,True,passed,45.0309,2504.0,5.0,True,0.2055,1.8242,0.0,0.0,False
4,artifact_5,Pandas,True,passed,96.2912,4079.0,94.0,True,0.2059,1.8242,0.0,0.0,False
5,artifact_6,NumPy,True,passed,55.2295,2799.0,37.0,True,0.2053,1.8203,0.0,0.0,False
6,artifact_7,Matplotlib,True,passed,100.6347,4919.0,69.0,True,0.2061,6.2383,0.0,0.0,False
7,artifact_8,Scrapy,True,passed,7.2152,823.0,18.0,True,0.2060,1.8242,0.0,0.0,False
8,artifact_9,Flask,True,passed,2.8905,289.0,6.0,True,0.2047,1.8281,0.0,0.0,False
9,artifact_10,TensorFlow,True,passed,567.3861,39389.0,0.0,True,0.2062,6.1055,0.0,0.0,False


In [6]:
output_file = "resource_efficiency_results.json"

with open(output_file, "w", encoding="utf-8") as file:
    json.dump(results, file, indent=4)

print(f"✅ Results saved to {output_file}")

✅ Results saved to resource_efficiency_results.json
